# ReAct Visualized: What the Model Actually Sees

**Prerequisite:** finish notebooks 01–02 first (and optionally `04_react_loop.ipynb`).

You built the ReAct loop in notebook 02. But what does the model actually receive as input on each iteration? How does the prompt grow? How many tokens does each step cost?

This notebook instruments the loop so you can see everything.

In [ ]:
!uv pip install -q openai tiktoken python-dotenv requests

In [ ]:
# ── Setup (same as notebook 02 — see that notebook for explanations) ──
import os, re, requests
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()

REACT_PROMPT = """Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
{scratchpad}"""

def add(expression: str) -> str:
    a, b = [int(x.strip()) for x in expression.split(",")]
    return str(a + b)

def multiply(expression: str) -> str:
    a, b = [int(x.strip()) for x in expression.split(",")]
    return str(a * b)

def web_search(query: str) -> str:
    api_key = os.getenv("TAVILY_API_KEY")
    if not api_key:
        return f"[FAKE SEARCH] pretend results for: {query}"
    r = requests.post("https://api.tavily.com/search",
                      json={"api_key": api_key, "query": query, "max_results": 3}, timeout=15)
    r.raise_for_status()
    return "\n".join(f"- {i['title']}: {i['content'][:200]}" for i in r.json().get("results", []))

TOOLS = {
    "add": (add, "Add two integers. Input: two integers separated by a comma, e.g. '17, 25'"),
    "multiply": (multiply, "Multiply two integers. Input: two integers separated by a comma, e.g. '6, 7'"),
    "web_search": (web_search, "Search the web for current information. Input: a plain search query string"),
}
tools_text = "\n".join(f"{n}: {d}" for n, (_, d) in TOOLS.items())
tool_names = ", ".join(TOOLS.keys())

def parse_output(text):
    final = re.search(r"Final Answer:\s*(.*)", text, re.DOTALL)
    if final:
        return {"final_answer": final.group(1).strip()}
    action = re.search(r"Action:\s*(.*?)(?:\n|$)", text)
    action_input = re.search(r"Action Input:\s*(.*)", text, re.DOTALL)
    if action and action_input:
        return {"action": action.group(1).strip(), "action_input": action_input.group(1).strip()}
    raise ValueError(f"Could not parse:\n{text}")

print("Setup complete.")

## 1. Counting tokens

Before we run the loop, we need a way to count tokens. The model does not see characters — it sees tokens. We use `tiktoken`, the same tokenizer OpenAI uses.

In [ ]:
import tiktoken

enc = tiktoken.encoding_for_model("gpt-4o-mini")

def count_tokens(text: str) -> int:
    return len(enc.encode(text))

# Quick demo
sample = "What is 12 multiplied by 7, then add 100 to that?"
print(f"Text: {sample!r}")
print(f"Tokens: {count_tokens(sample)}")
print(f"Characters: {len(sample)}")
print(f"Ratio: ~{len(sample) / count_tokens(sample):.1f} chars per token")

## 2. The instrumented loop

This is the same ReAct loop from notebook 02, but now it records and prints:
- The **full prompt** sent to the model at each step
- The **number of tokens** in that prompt
- The **model's response**
- The **scratchpad** after appending the observation
- A running **token cost table**

In [ ]:
def react_agent_visualized(question: str, max_steps: int = 8):
    scratchpad = ""
    step_data = []  # collect metrics

    for step in range(1, max_steps + 1):
        prompt = REACT_PROMPT.format(
            tools=tools_text, tool_names=tool_names,
            input=question, scratchpad=scratchpad,
        )

        prompt_tokens = count_tokens(prompt)

        # ── Show the full prompt ──
        print(f"\n{'#'*70}")
        print(f"#  STEP {step} — FULL PROMPT SENT TO MODEL ({prompt_tokens} tokens)")
        print(f"{'#'*70}")
        print(prompt)
        print(f"{'\u2500'*70}")

        # ── Call the model ──
        response = client.chat.completions.create(
            model="gpt-4o-mini", temperature=0,
            stop=["\nObservation:", "Observation:"],
            messages=[{"role": "user", "content": prompt}],
        )
        text = response.choices[0].message.content
        completion_tokens = response.usage.completion_tokens
        total_usage = response.usage.total_tokens

        print(f"\nMODEL RESPONSE ({completion_tokens} tokens):")
        print(text)

        # Record metrics
        step_data.append({
            "step": step,
            "prompt_tokens": prompt_tokens,
            "completion_tokens": completion_tokens,
            "total_tokens": total_usage,
            "scratchpad_chars": len(scratchpad),
        })

        # ── Parse ──
        parsed = parse_output(text)

        if "final_answer" in parsed:
            print(f"\n>>> FINAL ANSWER: {parsed['final_answer']}")
            break

        # ── Run tool ──
        action = parsed["action"]
        action_input = parsed["action_input"]
        fn, _ = TOOLS[action]
        observation = fn(action_input)

        print(f"\n  Tool: {action}({action_input!r}) => {observation}")

        scratchpad += f"{text}\nObservation: {observation}\n"

        print(f"\n  Scratchpad is now {len(scratchpad)} chars / {count_tokens(scratchpad)} tokens")

    return step_data

In [ ]:
data = react_agent_visualized("What is 12 multiplied by 7, then add 100 to that?")

## 3. Token cost table

Look at how the prompt grows at each step. The scratchpad accumulates all previous thoughts, actions, and observations — so each step sends *everything again* plus new content.

In [ ]:
print(f"{'Step':<6} {'Prompt tokens':<16} {'Completion tokens':<20} {'Total tokens':<14} {'Scratchpad chars':<18}")
print("\u2500" * 74)
cumulative = 0
for d in data:
    cumulative += d["total_tokens"]
    print(f"{d['step']:<6} {d['prompt_tokens']:<16} {d['completion_tokens']:<20} {d['total_tokens']:<14} {d['scratchpad_chars']:<18}")
print("\u2500" * 74)
print(f"{'SUM':<6} {'':<16} {'':<20} {cumulative:<14}")
print(f"\nTotal tokens consumed across all steps: {cumulative}")

## 4. The growth problem

Notice: the prompt tokens **increase** at every step because the scratchpad gets bigger. The model re-reads the entire conversation history on every iteration.

If the agent takes N steps:
- Step 1 sends the base prompt (~P tokens)
- Step 2 sends P + step_1_content
- Step 3 sends P + step_1_content + step_2_content
- ...

Total tokens consumed = P×N + (content accumulated across all steps, re-sent each time)

This is why long-running agents get expensive fast. Let's see it with a longer question.

In [ ]:
data2 = react_agent_visualized("Multiply 15 by 3, then add 200 to that, then add 50 more to that result.")

In [ ]:
print("\nToken growth per step:")
print()
for d in data2:
    bar = "\u2588" * (d["prompt_tokens"] // 10)
    print(f"  Step {d['step']}: {d['prompt_tokens']:>5} tokens  {bar}")
print()
print("Each bar represents prompt tokens. Notice it grows with every step.")
print("This is the cost of carrying the full scratchpad in context.")

## 5. Prompt diff: Step 1 vs Step 3

To make the growth concrete, let's render the prompt at step 1 and the prompt at step 3 side by side — the only difference is the scratchpad section.

In [ ]:
def get_prompt_at_step(question, step_num, max_steps=8):
    """Run the agent and capture the prompt at a specific step."""
    scratchpad = ""
    prompts = {}
    for step in range(1, max_steps + 1):
        prompt = REACT_PROMPT.format(
            tools=tools_text, tool_names=tool_names,
            input=question, scratchpad=scratchpad,
        )
        prompts[step] = prompt
        if step == step_num:
            return prompts

        response = client.chat.completions.create(
            model="gpt-4o-mini", temperature=0,
            stop=["\nObservation:", "Observation:"],
            messages=[{"role": "user", "content": prompt}],
        )
        text = response.choices[0].message.content
        parsed = parse_output(text)
        if "final_answer" in parsed:
            return prompts
        fn, _ = TOOLS[parsed["action"]]
        observation = fn(parsed["action_input"])
        scratchpad += f"{text}\nObservation: {observation}\n"
    return prompts

q = "Multiply 15 by 3, then add 200 to that, then add 50 more to that result."
prompts = get_prompt_at_step(q, 3)

for step_num, prompt_text in prompts.items():
    print(f"\n{'='*70}")
    print(f"PROMPT AT STEP {step_num} ({count_tokens(prompt_text)} tokens)")
    print(f"{'='*70}")
    print(prompt_text)

## 6. Why this matters for production

The scratchpad is a **growing context window**. Every step adds ~50–200 tokens of history. For an agent that takes 10 steps, you are paying for:

| Step | Prompt size |
|------|------------|
| 1 | P |
| 2 | P + ~100 |
| 3 | P + ~200 |
| ... | ... |
| 10 | P + ~900 |

Total input tokens ≈ **10P + 4,500** instead of just **P + 900** if you only sent the latest context.

This is why production agent systems use techniques like:
- **Summarizing** older scratchpad entries instead of keeping them verbatim
- **Sliding windows** that only keep the last K steps
- **Message pruning** to drop irrelevant earlier steps
- **Native tool calling** with message-based context (where the API manages the conversation)

The ReAct text-scratchpad approach is the simplest to understand but the most expensive to run.

**See also:** `03_react_failure_modes.ipynb` — we deliberately break each component of the loop and watch what happens.